# 45 — A2A Agent Handoff

**What you'll learn in this notebook:**

1. How to pass structured Pydantic objects between agents (not raw strings)
2. Why typed contracts beat raw strings for inter-agent communication
3. How to use `with_structured_output` for agent-to-agent communication
4. The planner / executor / synthesizer pattern for decomposed task execution

**Context:** A2A (Agent-to-Agent) protocol is an emerging standard for inter-agent communication (Google, 2025). This example teaches the typed handoff primitive — the building block that lets agents coordinate reliably without parsing unstructured text.

In [ ]:
# Install dependencies (Colab guard)
import sys

if "google.colab" in sys.modules:
    !pip install -q langchain langchain-openai langgraph pydantic python-dotenv

In [ ]:
import os

# Set API key — Colab secrets or environment variable
if "google.colab" in sys.modules:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv

    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running"

In [ ]:
# --- Concept: Agent Communication Evolution ---
#
# Option 1: Raw string  -> fragile, unstructured, hard to validate
#   planner_output = "Research vector databases and write a summary"
#   executor_input = planner_output  # just a string -- what fields exist?
#
# Option 2: Dict  -> better, but no schema enforcement
#   planner_output = {"instruction": "...", "context": "..."}  # no validation
#
# Option 3: Pydantic model  -> validated contract, serializable, self-documenting
#   class AgentTask(BaseModel):
#       task_id: str        # forces naming
#       task_type: str      # forces classification
#       instruction: str    # forces explicit directive
#       context: str        # forces background info
#       expected_output: str  # forces output spec
#
# The Pydantic approach gives you:
#   - Automatic validation on creation
#   - .model_dump() for serialization to state dict
#   - Field descriptions that guide the LLM (via with_structured_output)
#   - Type safety for downstream agents

print("A2A: agents communicate via typed contracts")
print("AgentTask fields: task_id, task_type, instruction, context, expected_output")

In [ ]:
from typing import Optional

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field
from typing_extensions import TypedDict


# --- Pydantic handoff contract ---
class AgentTask(BaseModel):
    """Structured handoff from Planner to Executor."""

    task_id: str = Field(description="Unique identifier for this task")
    task_type: str = Field(description="Type: research, analysis, or synthesis")
    instruction: str = Field(description="Clear instruction for the executor")
    context: str = Field(description="Background context to help executor")
    expected_output: str = Field(description="What the executor should produce")


# --- Shared graph state ---
class A2AState(TypedDict):
    original_request: str
    task_dict: Optional[dict]  # serialized AgentTask
    execution_result: str
    final_synthesis: str


# --- LLMs ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
planner_llm = llm.with_structured_output(AgentTask)

# --- Prompts ---
PLANNER_PROMPT = """You are a planning agent. Given a request, create a structured task for an executor agent.

Request: {request}

Create a task with: task_id (short slug), task_type (research/analysis/synthesis), instruction (clear directive), context (relevant background), expected_output (what executor should return)."""

EXECUTOR_PROMPT = """You are an executor agent. Complete the task below thoroughly.

Task: {instruction}
Context: {context}
Expected Output: {expected_output}

Provide a clear, well-structured response."""

SYNTHESIZER_PROMPT = """You are a planning agent finalizing a request.

Original Request: {original_request}
Task Created: {task_instruction}
Executor Result: {execution_result}

Synthesize a polished final answer that addresses the original request."""


# --- Nodes ---
def planner_agent(state: A2AState) -> dict:
    """Agent A: plans the task and creates structured AgentTask handoff."""
    prompt = PLANNER_PROMPT.format(request=state["original_request"])
    task = planner_llm.invoke([HumanMessage(content=prompt)])
    print(f"  Planner created task: {task.task_id} ({task.task_type})")
    print(f"  Instruction: {task.instruction[:80]}...")
    return {"task_dict": task.model_dump()}


def executor_agent(state: A2AState) -> dict:
    """Agent B: receives structured task and executes it."""
    task = state["task_dict"]
    prompt = EXECUTOR_PROMPT.format(
        instruction=task["instruction"],
        context=task["context"],
        expected_output=task["expected_output"],
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    print(f"  Executor completed task: {len(result.content)} chars")
    return {"execution_result": result.content}


def synthesizer_agent(state: A2AState) -> dict:
    """Agent A synthesizes executor output into final answer."""
    task = state["task_dict"]
    prompt = SYNTHESIZER_PROMPT.format(
        original_request=state["original_request"],
        task_instruction=task["instruction"],
        execution_result=state["execution_result"],
    )
    result = llm.invoke([HumanMessage(content=prompt)])
    return {"final_synthesis": result.content}


# --- Build graph ---
def create_workflow():
    graph = StateGraph(A2AState)
    graph.add_node("planner", planner_agent)
    graph.add_node("executor", executor_agent)
    graph.add_node("synthesizer", synthesizer_agent)
    graph.set_entry_point("planner")
    graph.add_edge("planner", "executor")
    graph.add_edge("executor", "synthesizer")
    graph.add_edge("synthesizer", END)
    return graph.compile()


app = create_workflow()
print("Workflow compiled: planner -> executor -> synthesizer -> END")

In [ ]:
SAMPLE_REQUESTS = [
    "Research the key benefits of vector databases and write a summary.",
    "Explain the CAP theorem and its practical implications.",
]

for request in SAMPLE_REQUESTS:
    print(f"\nREQUEST: {request}")
    print("─" * 60)
    result = app.invoke(
        {
            "original_request": request,
            "task_dict": None,
            "execution_result": "",
            "final_synthesis": "",
        }
    )
    task = result["task_dict"]
    print(f"\nTask ID   : {task['task_id']}")
    print(f"Task Type : {task['task_type']}")
    print(f"Instruction preview: {task['instruction'][:120]}...")
    print(f"Execution result length: {len(result['execution_result'])} chars")
    print(f"\nFinal Synthesis (preview):\n{result['final_synthesis'][:400]}")
    print()

## Exercises

1. **Add a `priority` field** — extend `AgentTask` with `priority: Literal["low", "medium", "high"]` and test that the planner assigns priorities correctly based on the request urgency.

2. **Add a validator** — use a Pydantic `@field_validator` on `task_type` that raises `ValueError` if the value is not in `["research", "analysis", "synthesis"]`. Observe how `with_structured_output` handles validation errors.

3. **Add a QA agent node** — insert a `qa_agent` node between `executor` and `synthesizer` that scores the executor's result 1-10 and either approves it or sends it back for revision (use a conditional edge).

4. **Make it async** — convert the workflow to use `await app.ainvoke()` and run both requests concurrently with `asyncio.gather()`.

---

## What's next?

- **[6-multi-agent-graph](../6-multi-agent-graph/)** — subgraph composition with shared state
- **[43-supervisor-worker](../43-supervisor-worker/)** — dynamic routing between specialized workers
- **[38-langgraph-command-pattern](../38-langgraph-command-pattern/)** — Command-based handoffs for fine-grained flow control